# Session 3

## Building Agentic LLM Workflows for Biomedical Knowledge Graphs

In Session 2, you followed a workflow that had been defined in advance: the notebook specified which operations to run, and in what order, to perform a MOFA analysis. In this session, we make this workflow **dynamic**. Instead of specifying the sequence of analysis steps beforehand, we give a language model access to functions for interrogating a pre-fitted MOFA model. The model can then choose which function to call, use the result to decide what information it needs next, and continue until it can answer the question.

A function made available to a language model in this way is called a **tool**. The program around the model that actually executes those tool calls is often called a **harness**. When the model can choose tools, use their results, and repeat this process until it can answer a question, the resulting system is an **LLM agent**.

Tools determine what an agent can do, but there are other ways to shape how those capabilities are made available and used. **MCP**, the Model Context Protocol, provides a standard way for an agent to connect to tools and data sources, while **Skills** provide reusable instructions for how an agent should carry out a task.

### What is an LLM agent?

A language model such as GPT-5 or Claude cannot, by itself, execute functions or access your files. To interact with code or data, it needs a program around it — the **harness** — that can perform those actions on its behalf.

For example, the harness might tell Claude that a function called `top_features_for_factor` is available, what arguments it takes, and what it returns. Claude can then request that function, the Python code runs it on the fitted MOFA model, and the result isreturned to Claude as new context.

An LLM agent combines the flexibility of a language model with the reliability of ordinary analysis code. A hard-coded pipeline is predictable and easy to trace, but it can only follow the workflow defined in advance. A standalone language model is much more flexible about the questions it can understand and the answers it can formulate, but without access to the analysis it cannot inspect the MOFA results on its own and may generate claims that are not supported by the data.

With an agent, the model can decide which analysis steps are useful for a particular question, while the calculations themselves are still performed by ordinary code on the real data. The resulting tool outputs can therefore be inspected and traced back to the functions that produced them.

The trade-off is that the **route through the analysis is no longer fixed**. Two runs may choose different tools, call them in a different order, or take a different number of steps before answering. Each additional model call also adds time and API cost.

For the model to use the result of one tool call to decide what to do next, the harness has to call the model repeatedly. This repeated exchange between the model and the harness is called the **agent loop**. 

On each turn, the model sees the original question, some predefined instructions — the **system prompt** — and any results gathered so far. It can then either produce a final answer or request one or more tools.

If it requests a tool, the harness executes the corresponding function, adds the result to the conversation, and calls them model again. The model now has new information on which to base its next decision: it may request another tool or decide that it has enough evidence to answer.

This cycle continues until the model produces a final answer, or until the harness stops it at a predefined limit such as a maximum number of steps or tokens.

This session looks at three complementary parts of an agentic system: *tools* define what the agent can do, *MCP* standardises how capabilities are made available, and *Skills* guide how those capabilities should be used.

1. **Tools** give the model specific capabilities. A tool has a name, a set of arguments, and a description of what it does. The model uses that information to decide whether the tool is useful; if it requests the tool, the harness runs the underlying function and returns the result. In Part 1, we give an LLM access to tools for interrogating a fitted MOFA model.
2. **MCP**, the Model Context Protocol, provides a standard way for agents to connect to tools and other resources. Instead of defining every capability separately into each application, an *MCP server* can expose them through a common interface that compatible agents can discover and use. The server can run locally or remotely, making the same capabilities easier to reuse across applications and allowing agents to connect to tools developed by others. In Part 2, we first expose our own MOFA functions through MCP and then access an external MCP (BioMCP), which provides access to several public biomedical resources.
    
3. **Skills** provide instructions for how an agent should approach a task rather than giving it new capabilities. A skill can describe which tools or resources to use, which steps matter, what should count as evidence, what should be avoided, and how the result should be reported. The skill itself does not execute anything. Instead, it guides how the agent uses the tools and other capabilities already available to it.

### What's in this session
Throughout the session, we work with the same TCGA breast-cancer multi-omics data and a pre-fitted MOFA model.

Parts 1 and 2 use the same main question:

> Which MOFA factor is most associated with breast-cancer subtype, and which transcriptomic features most strongly drive it?

They also use the same underlying MOFA analysis functions. This lets us compare two ways of giving an agent access to tools: in Part 1, they are called directly; in Part 2, the same functions are made available through MCP.

Part 3 keeps the same data and biological setting, but changes the agent's capabilities. Instead of giving it a fixed set of MOFA-specific tools, we use a coding agent with general-purpose tools and examine how a **skill** can guide the way it carries out the analysis.

### Part 0 — Data reconciliation
The latter analyses assumes that the tree omics tables describe the same patients in the same order, and the agent, as it is structured, has no way to notice when they do not. The raw tables, however, cover overlapping but different sets of patients. This notebook keeps the 603 patients measured in all three, checks that the tables agree on each patient's subtype, and writes the single aligned table the rest of the session reads.

### Part 1 — MOFA tools oforan LLM agent
Part 1 introduces the simplest setup: ordinary Python functions from `mofa_tools.py` are made available to Claude as **tools** using a library called `LangChain`. Claude sees what each tool does and what arguments it takes, and decides which one to call and in what order.

We also build the **agent loop** explicitly. Claude receives a question, requests a tool to be run, and checks the results, the process repeating until Claude has enough information to answer. We start with questions that require a single tool and then move to questions that require several tools to be combined.

**Key takeaways**
- Claude sees a tool's interface — its name, arguments, and description. It does not read how it is implemented
- Claude chooses which tools to call and in what order; the calculations themselves are still performed by Python on the fitted MOFA model.

### Part 2 — MOFA tools over the Model Context Protocol (MCP)

Part 2 keeps the same MOFA functions and the same question but changes **how the agent reaches those functions**. 

In Part 1, the harness calls the MOFA functions directly from the Python code. In Part 2, we make those same functions available through a separate programme. The harness no longer imports the functions itself: instead, it asks that program what tools are available, requests to run one when Claude wants to use it, and receives the results back.

We call that separate program a **server**. To make this useful, the harness and the server need to agree on how requests and results are exchanged. **MCP — the Model Context Protocol — defines that common way of communicating**. It specifies, for example, how the agent can ask a server which tools it provides and how it can request that one of those tools be run.

We first build our own MCP server around the MOFA tools, so we can see how this works with functions we already know. We then see why this becomes useful in practice by connecting the same agent to an external MCP server written by someone else. Here, we use *BioMCP*, which adds tools for searching biomedical resources such as PubMed, ClinicalTrials.gov, and ClinVar. The agent can then use these tools alongside our own MOFA tools within the same workflow, without us having to write these functions ourselves.

**Key takeaways**
- A MCP server is a program that makes tools available to other applications
- MCP defines a standard way for those applications to discover and call the tools
- Because different servers follow the same standard, an agent can combine tools from our own code with tools provided by others.

### Part 3 — Agent Skills: teaching an agent how, not just what

Parts 1 and 2 both hand Claude a fixed menu of domain-specific tools. Part 3 changes the setup: we give the agent general-purpose tools for reading files, running commands and writing files. With those capabilities, the agent can find an drun the MOFA analysis itself. What it does not automatically know is how we want the task to be carried out: which model to load, what should could as evidence, which steps matter, and how the result should be reporeted.

An **Agent Skill** provides that procedural guidance. Here, the skill tells the agent how to work with the MOFA analysis. We run the same task with and without the skill: what the agent can do stays the same, while the instructions guiding how it does it change.

**Key takeaways**
- Tools give an agent capabilities, a skill gives it instructions for how to use those capabilities for a particular task.
- The same agent can therefore behave differently with and without a skill, even though its underlying tools have not changed
- Rules such as "use the cached model rather than refitting it" are examples of procedural knowledge that can be encoded in a skill.